# Optimising Plate and Cone Position

In [1]:
import sys
from pathlib import Path

# adjust the number of .parent calls to match your notebook's depth from repo root
repo_root = Path.cwd().resolve().parents[1]   # e.g. if notebook is 2 levels deep
sys.path.append(str(repo_root))

from molflow_sweep import FacetRef, SweepConfig, MolflowSweep

import optuna
# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization.matplotlib import plot_contour
from optuna.visualization.matplotlib import plot_edf
from optuna.visualization.matplotlib import plot_intermediate_values
from optuna.visualization.matplotlib import plot_optimization_history
from optuna.visualization.matplotlib import plot_parallel_coordinate
from optuna.visualization.matplotlib import plot_param_importances
from optuna.visualization.matplotlib import plot_rank
from optuna.visualization.matplotlib import plot_slice
from optuna.visualization.matplotlib import plot_timeline


c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

PLATE = ["plate_front", "plate_back"]


#cone has 36 facets, plus front and back, so 38 in total
CONE = ["cfront","cback","c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c9", "c10","c11", "c12", "c13", "c14", "c15", "c16", "c17", "c18", "c19", "c20","c21", "c22", "c23", "c24", "c25", "c26", "c27", "c28", "c29", "c30", "c31", "c32", "c33", "c34", "c35", "c36"]

cfg = SweepConfig(
    molflow_exe=r"C:\Users\ASUS\Documents\molflow_win_2.11.1\molflowCLI.exe",
    base_geometry=str(repo_root / "geometries" / "simplified_target_baseplate_cone.xml"),
    out_dir=Path.cwd().resolve() / "sweep_runs",

    facets={
        "plate_front": FacetRef("plate_front", xml_id=175, csv_id=176),
        "plate_back":  FacetRef("plate_back", xml_id=176, csv_id=177),
        "deflector": FacetRef("deflector", xml_id=252, csv_id=253),
        "cfront": FacetRef("cfront", xml_id=255, csv_id=256),
        "cback": FacetRef("cback", xml_id=256, csv_id=257),
        "c1": FacetRef("c1", xml_id=257, csv_id=258),
        "c2": FacetRef("c2", xml_id=258, csv_id=259),
        "c3": FacetRef("c3", xml_id=259, csv_id=260),
        "c4": FacetRef("c4", xml_id=260, csv_id=261),
        "c5": FacetRef("c5", xml_id=261, csv_id=262),
        "c6": FacetRef("c6", xml_id=262, csv_id=263),
        "c7": FacetRef("c7", xml_id=263, csv_id=264),
        "c8": FacetRef("c8", xml_id=264, csv_id=265),
        "c9": FacetRef("c9", xml_id=265, csv_id=266),
        "c10": FacetRef("c10", xml_id=266, csv_id=267),
        "c11": FacetRef("c11", xml_id=267, csv_id=268),
        "c12": FacetRef("c12", xml_id=268, csv_id=269),
        "c13": FacetRef("c13", xml_id=269, csv_id=270),
        "c14": FacetRef("c14", xml_id=270, csv_id=271),
        "c15": FacetRef("c15", xml_id=271, csv_id=272),
        "c16": FacetRef("c16", xml_id=272, csv_id=273),
        "c17": FacetRef("c17", xml_id=273, csv_id=274),
        "c18": FacetRef("c18", xml_id=274, csv_id=275),
        "c19": FacetRef("c19", xml_id=275, csv_id=276),
        "c20": FacetRef("c20", xml_id=276, csv_id=277),
        "c21": FacetRef("c21", xml_id=277, csv_id=278),
        "c22": FacetRef("c22", xml_id=278, csv_id=279),
        "c23": FacetRef("c23", xml_id=279, csv_id=280),
        "c24": FacetRef("c24", xml_id=280, csv_id=281),
        "c25": FacetRef("c25", xml_id=281, csv_id=282),
        "c26": FacetRef("c26", xml_id=282, csv_id=283),
        "c27": FacetRef("c27", xml_id=283, csv_id=284),
        "c28": FacetRef("c28", xml_id=284, csv_id=285),
        "c29": FacetRef("c29", xml_id=285, csv_id=286),
        "c30": FacetRef("c30", xml_id=286, csv_id=287),
        "c31": FacetRef("c31", xml_id=287, csv_id=288),
        "c32": FacetRef("c32", xml_id=288, csv_id=289),
        "c33": FacetRef("c33", xml_id=289, csv_id=290),
        "c34": FacetRef("c34", xml_id=290, csv_id=291),
        "c35": FacetRef("c35", xml_id=291, csv_id=292),
        "c36": FacetRef("c36", xml_id=292, csv_id=293),
    },
    move_facets=CONE,
    result_facets=PLATE + CONE + ["deflector"],
    axis=0,
    ndes="1e5",
    threads="8",
)

sweep = MolflowSweep(cfg)

sweep = MolflowSweep(cfg)

In [3]:
def objective(trial):
    # Suggest values for the parameters to optimize
    plate_off = trial.suggest_float('plate_offset', -0.25, 10.0)  # Plate offset between -0.25 and 10.0 cm
    cone_off = trial.suggest_float('cone_offset', -0.75, 0.75)  # Cone offset between -0.25 and 0.75 cm

    # Evaluate the sweep with the suggested parameters
    row = sweep.evaluate(
        moves=[(PLATE, 0, plate_off),   # facets, axis, offset
               (CONE, 0, cone_off+plate_off)],  # Adjust cone offset relative to plate
        run_id=f"Plate_{plate_off:.3f}_Cone_{cone_off:.3f}")
    

    # Return the objective value (hit fraction on back of plate)
    return row['plate_back_equiv_abs'] / row['total_des']  # Hit fraction on back of plate  

In [4]:
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.GPSampler(),
                            )  # We want to maximize the fraction that hit the back plate 


C:\Users\ASUS\AppData\Local\Temp\ipykernel_28664\2331773744.py:2: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler=optuna.samplers.GPSampler(),
[I 2026-07-22 17:22:33,778] A new study created in memory with name: no-name-acde0200-8e4b-48cd-87de-bf745e9eea95


In [1]:
study.optimize(objective, n_trials=5)  # Run 5 trials

NameError: name 'study' is not defined

In [ ]:
study.optimize(objective, n_trials=25)  # Run 25 trials

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_contour(study)

In [ ]:
plot_param_importances(study)

In [ ]:
print("Best params: ", study.best_params)
print("Best value: ", study.best_value)
print("Best Trial: ", study.best_trial)
print("Trials: ", study.trials)